<a href="https://colab.research.google.com/github/re1nsilitonga/LearnAI-ML/blob/main/SpotifyAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import libraries

In [ ]:
import pandas as pd
import numpy as np

# Stage 1: Setup and Initial Inspection

## Load the dataset

In [ ]:
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-01-21/spotify_songs.csv'
df = pd.read_csv(url)

df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


## Perform initial inspection
*   the shape
*   dtypes
*   the number of missing values per column.

In [ ]:
df.shape

(32833, 23)

In [ ]:
df.dtypes

,0
track_id,object
track_name,object
track_artist,object
track_popularity,int64
track_album_id,object
track_album_name,object
track_album_release_date,object
playlist_name,object
playlist_id,object
playlist_genre,object


In [ ]:
df.isnull().sum()

,0
track_id,0
track_name,5
track_artist,5
track_popularity,0
track_album_id,0
track_album_name,5
track_album_release_date,0
playlist_name,0
playlist_id,0
playlist_genre,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32833 entries, 0 to 32832
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  32833 non-null  object 
 1   track_name                32828 non-null  object 
 2   track_artist              32828 non-null  object 
 3   track_popularity          32833 non-null  int64  
 4   track_album_id            32833 non-null  object 
 5   track_album_name          32828 non-null  object 
 6   track_album_release_date  32833 non-null  object 
 7   playlist_name             32833 non-null  object 
 8   playlist_id               32833 non-null  object 
 9   playlist_genre            32833 non-null  object 
 10  playlist_subgenre         32833 non-null  object 
 11  danceability              32833 non-null  float64
 12  energy                    32833 non-null  float64
 13  key                       32833 non-null  int64  
 14  loudne

Based on df.info(), each column contains 32,833 rows. Since df.isnull().sum() indicates a maximum of only 5 missing values per column, dropping columns is not required

## Identify and handle duplicate rows

In [ ]:
duplicate_sum = df.duplicated().sum()
print(f'the number of duplicate row : {duplicate_sum}')

df.drop_duplicates(inplace=True)
duplicate_sum = df.duplicated().sum()
print(f'the number of duplicate row : {duplicate_sum}')

the number of duplicate row : 0
the number of duplicate row : 0


There is no duplicate row

## Critical note: track-level duplication (beyond the full-row check)

*  df.duplicated() only catches EXACT duplicate rows.
*  A track can legitimately appear in multiple playlists (different playlist_id / playlist_name / playlist_genre)
*  So the same track_id can repeat without the row itself being a full duplicate.

In [ ]:
n_unique_tracks = df['track_id'].nunique()
n_dup_track_rows = df.duplicated(subset=['track_id']).sum()

print(f"Total rows: {len(df)}")
print(f"Unique track_id: {n_unique_tracks}")
print(f"Rows with a repeated track_id: {n_dup_track_rows}")

example_id = df[df.duplicated(subset=['track_id'], keep=False)]['track_id'].value_counts().index[0]
print()
print(df[df['track_id'] == example_id][['track_id', 'track_name', 'track_artist', 'playlist_name', 'playlist_genre']].to_string(index=False))

Total rows: 32833
Unique track_id: 28356
Rows with a repeated track_id: 4477

              track_id            track_name     track_artist                                                                                  playlist_name playlist_genre
7BKLCZ1jbUBVqRi2FVlTVw Closer (feat. Halsey) The Chainsmokers                                                                                      Dance Pop            pop
7BKLCZ1jbUBVqRi2FVlTVw Closer (feat. Halsey) The Chainsmokers                                                                                  Post pop teen            pop
7BKLCZ1jbUBVqRi2FVlTVw Closer (feat. Halsey) The Chainsmokers                                                                     Electropop Hits  2017-2020            pop
7BKLCZ1jbUBVqRi2FVlTVw Closer (feat. Halsey) The Chainsmokers                                                          A Loose Definition of Indie Poptimism            pop
7BKLCZ1jbUBVqRi2FVlTVw Closer (feat. Halsey) The Chainsmokers 

**The full-row check is correct, but incomplete.**

* `df.duplicated()` correctly returns 0 because `playlist_id` and `playlist_name` differ between rows.
* Every row is a distinct (track, playlist) pairing, so a full-row check cannot catch this kind of repetition.
* At the `track_id` level, **4,477 rows** share a `track_id` with at least one other row.
* Example: *Closer (feat. Halsey)* by The Chainsmokers appears **10 times**, spread across `pop`, `rap`, `latin`, `r&b`, and `edm` playlists.
* Only **28,356** of the 32,833 rows are actually unique tracks.

This is not a data quality bug. It reflects the real Spotify behaviour of the same song being placed into many playlists, and even tagged under different genres. That said, it does change how some later analyses should be handled.

* **Stage 2's per `playlist_genre` groupby is fine to keep as is.** Each row legitimately represents that track's placement inside a specific genre's playlist. Counting a track once per genre it appears in is intentional, not an error.
* **For dataset wide, per track analyses (Stage 3, Bonus 1)**, keeping every playlist duplicated row would silently over weight tracks that happen to sit in many playlists. This biases statistics such as the correlation matrix or an artist's "average sound" toward a handful of viral songs instead of reflecting the artist's or dataset's actual diversity.

**Decision:**

* From Stage 3 onward, track level numeric analysis uses `df_unique` (one row per `track_id`, first occurrence kept).
* Genre level analysis that intentionally reuses Stage 2's per genre logic (Bonus 2's genre centroids) continues to use the full `df`, since a track's placement in a genre's playlist is a legitimate signal for that genre's audio profile.


In [ ]:
df_unique = df.drop_duplicates(subset=['track_id'], keep='first').reset_index(drop=True)
print(df_unique.shape)

(28356, 23)


## Summary table

In [ ]:
# Get all numeric column names
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Create a list to store summary metrics
summary_data = []

for col in numeric_cols:
    # Drop missing values for accurate calculations
    col_data = df[col].dropna()

    # Calculate statistical metrics using NumPy
    mean_val = np.mean(col_data)
    median_val = np.median(col_data)

    # ddof=1 computes the Sample Standard Deviation (matching Pandas/R default)
    std_val = np.std(col_data, ddof=1)

    # MANUAL IQR CALCULATION
    # Get the 75th (Q3) and 25th (Q1) percentiles
    q75, q25 = np.percentile(col_data, [75, 25])
    iqr_val = q75 - q25

    # Append results to the list
    summary_data.append({
        'Column Name': col,
        'Mean': mean_val,
        'Median': median_val,
        'Standard Deviation': std_val,
        'IQR': iqr_val
    })

# Convert the list of dictionaries into a clean DataFrame table
summary_table = pd.DataFrame(summary_data)

summary_table

,Column Name,Mean,Median,Standard Deviation,IQR
0,track_popularity,42.477081,45.000000,24.984074,38.00000
1,danceability,0.654850,0.672000,0.145085,0.19800
2,energy,0.698619,0.721000,0.180910,0.25900
3,key,5.374471,6.000000,3.611657,7.00000
4,loudness,-6.719499,-6.166000,2.988436,3.52600
5,mode,0.565711,1.000000,0.495671,1.00000
6,speechiness,0.107068,0.062500,0.101314,0.09100
7,acousticness,0.175334,0.080400,0.219633,0.23990
8,instrumentalness,0.084747,0.000016,0.224230,0.00483
9,liveness,0.190176,0.127000,0.154317,0.15530


# Stage 2 Genre and Popularity Analysis

## Compute the distribution statistics of track_popularity, danceability, energy, and valence per playlist_genre.

In [ ]:
# Select the target columns and group by playlist_genre
columns_gen_pop = ['track_popularity', 'danceability', 'energy', 'valence']

genre_stats = (
    df.groupby('playlist_genre')[columns_gen_pop]
    .agg(['mean', 'std', 'median'])
)

genre_stats

track_popularity                   danceability            \
                           mean        std median         mean       std   
playlist_genre                                                             
edm                   34.833526  23.154235   36.0     0.655041  0.123558   
latin                 47.026576  25.424557   50.0     0.713287  0.114974   
pop                   47.744870  25.158331   52.0     0.639302  0.128221   
r&b                   41.223532  25.894504   44.0     0.670179  0.138213   
rap                   43.215454  23.302085   47.0     0.718353  0.136452   
rock                  41.728338  24.825230   46.0     0.520548  0.139863   

                         energy                    valence                   
               median      mean       std median      mean       std median  
playlist_genre                                                               
edm             0.659  0.802476  0.139415  0.830  0.400656  0.226205  0.370  
latin           0.729  0.708312  0.152308  0.729  0.605510  0.222289  0.628  
pop             0.652  0.701028  0.171084  0.727  0.503521  0.220472  0.500  
r&b             0.689  0.590934  0.179407  0.596  0.531231  0.225883  0.542  
rap             0.737  0.650708  0.170340  0.665  0.505090  0.224663  0.517  
rock            0.523  0.732813  0.194501  0.775  0.537352  0.229346  0.531

### **Answer:**

The genre with the highest variance in track popularity is **R&B** (with the highest standard deviation of **25.89**).

### **Business Perspective Interpretation (Content Recommendation):**

From a music recommendation system standpoint, high variance indicates that popularity within the R&B genre is highly spread out, meaning the genre contains a mix of both massive blockbuster hits and a deep "long-tail" of obscure or niche tracks.

Here is how a recommendation business should handle this:

1. **Avoid "Average" Biases:** Because the variance is high, recommending a track simply because it is an "average R&B track" is a weak strategy. The average popularity score (41.2) does not accurately represent the typical R&B song.
2. **Implement an Exploration vs. Exploitation Strategy:**
* *Exploitation:* Easily recommend the top-tier viral R&B hits to mainstream listeners.
* *Exploration:* Actively surface the large pool of low-popularity, niche R&B tracks to dedicated genre fans who look for hidden gems.


3. **Combine Collaborative and Content-Based Filtering:**
Relying solely on collaborative filtering (what other users listen to) will cause a feedback loop that only recommends the few hyper-popular R&B tracks. To successfully recommend across the high-variance spectrum, the algorithm must leverage **content-based filtering** (analyzing audio attributes like low energy or high valence) to match the deeper, less-popular tracks with users who have highly specific tastes.

## 10 artists with the most tracks

In [ ]:
top_10_artists = df['track_artist'].value_counts().head(10).index

# Filter dataset for only these 10 artists and calculate mean popularity
top_10_stats = (
    df[df['track_artist'].isin(top_10_artists)]
    .groupby('track_artist')['track_popularity']
    .agg(track_count='count', mean_popularity='mean')
    .sort_values(by='mean_popularity', ascending=False)
)

print(top_10_stats)

# Check overall correlation between volume (track count) and quality (mean popularity) for all artists
artist_counts_vs_pop = (
    df.groupby('track_artist')['track_popularity']
    .agg(track_count='count', mean_popularity='mean')
)
correlation = artist_counts_vs_pop['track_count'].corr(artist_counts_vs_pop['mean_popularity'])
print(f"\nOverall Correlation: {correlation}")

                           track_count  mean_popularity
track_artist                                           
Kygo                                83        65.120482
Calvin Harris                       91        61.813187
The Chainsmokers                   123        57.699187
David Guetta                       110        53.436364
Martin Garrix                      161        47.204969
Drake                              100        46.430000
Queen                              136        43.000000
Dimitri Vegas & Like Mike           93        42.795699
Don Omar                           102        41.950980
Hardwell                            84        39.107143

Overall Correlation: 0.1309008857481393


### **Key Insights from the Output**

* **Top Performer by Popularity:** **Kygo** has the highest mean track popularity (**65.12**) despite having one of the lowest track counts (83) among this group.
* **Top Performer by Volume:** **Martin Garrix** has the highest track volume (**161 tracks**), but sits in the middle tier for mean popularity (47.20).
* **Lowest Performer by Popularity:** **Hardwell** holds the lowest mean popularity score (**39.11**) with 84 tracks.

### **Correlation Analysis**

* **Overall Correlation Value:** **0.131**

#### **What this means:**

* There is a **very weak positive correlation** between the volume of tracks an artist releases and their average track popularity.
* **Business Takeaway:** Releasing more tracks (*volume*) does not translate to higher listener engagement or acclaim (*quality*). Popularity is driven by other critical factors such as marketing, genre trends, playlist placement, and individual song appeal rather than pure output quantity.

## Filter Tracks
*   track_popularity > 70
*   danceability > 0.7
*   energy > 0.6
*   duration_ms < 240000


In [ ]:
# Apply all filtering conditions
filtered_df = df[
    (df['track_popularity'] > 70) &
    (df['danceability'] > 0.7) &
    (df['energy'] > 0.6) &
    (df['duration_ms'] < 240000)
]

# Count how many tracks passed the filters
tracks_count = len(filtered_df)
print(f"Number of tracks that pass: {tracks_count}")

# 3. Find the dominant genre
if tracks_count > 0:
    genre_distribution = filtered_df['playlist_genre'].value_counts()
    dominant_genre = genre_distribution.idxmax()
    dominant_genre_count = genre_distribution.max()

    print(f"Dominant genre: '{dominant_genre}' with {dominant_genre_count} tracks.")
    print("\nFull Genre Distribution")
    print(genre_distribution)
else:
    print("No tracks met all the given criteria.")

Number of tracks that pass: 1112
Dominant genre: 'latin' with 396 tracks.

Full Genre Distribution
playlist_genre
latin    396
pop      264
rap      173
r&b      135
edm      111
rock      33
Name: count, dtype: int64


* **Number of tracks that pass:** 1,112 tracks
* **Dominant genre:** 'latin' (with 396 tracks)

# Stage 3: NumPy Analysis

## Extract and normalise the nine audio features

In [ ]:
feature_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
                'instrumentalness', 'liveness', 'valence', 'tempo']

def min_max_normalize(arr):
    col_min = arr.min(axis=0)
    col_max = arr.max(axis=0)
    return (arr - col_min) / (col_max - col_min)

feature_array = df_unique[feature_cols].to_numpy()
normalized_array = min_max_normalize(feature_array)

print("feature_array shape:", feature_array.shape)
print("normalized min per column:", normalized_array.min(axis=0).round(3))
print("normalized max per column:", normalized_array.max(axis=0).round(3))
print()
print("First 3 normalized rows:")
print(normalized_array[:3].round(3))

feature_array shape: (28356, 9)
normalized min per column: [0. 0. 0. 0. 0. 0. 0. 0. 0.]
normalized max per column: [1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 3 normalized rows:
[[0.761 0.916 0.918 0.064 0.103 0.    0.066 0.523 0.51 ]
 [0.739 0.815 0.869 0.041 0.073 0.004 0.358 0.699 0.418]
 [0.687 0.931 0.901 0.081 0.08  0.    0.11  0.619 0.518]]


Normalisation is fully vectorised.

* `arr.min(axis=0)` and `arr.max(axis=0)` operate on the whole 2D array at once, column by column.
* No explicit Python loop is used.
* No `sklearn.MinMaxScaler` is used.
* Every feature is now on the same [0, 1] scale, which also matters for Bonus 1's cosine similarity later on.


## Correlation matrix of the nine audio features

In [ ]:
corr_matrix = np.corrcoef(feature_array, rowvar=False)
corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)
print(corr_df.round(3))

mask_offdiag = ~np.eye(len(feature_cols), dtype=bool)
tri_mask = np.triu(mask_offdiag)
corr_masked = np.where(tri_mask, corr_matrix, np.nan)

max_idx = np.unravel_index(np.nanargmax(corr_masked), corr_masked.shape)
min_idx = np.unravel_index(np.nanargmin(corr_masked), corr_masked.shape)

print()
print(f"Highest positive correlation: {feature_cols[max_idx[0]]} & {feature_cols[max_idx[1]]} = {corr_matrix[max_idx]:.4f}")
print(f"Most negative correlation: {feature_cols[min_idx[0]]} & {feature_cols[min_idx[1]]} = {corr_matrix[min_idx]:.4f}")

                  danceability  energy  loudness  ...  liveness  valence  tempo
danceability             1.000  -0.081     0.015  ...    -0.127    0.334 -0.185
energy                  -0.081   1.000     0.682  ...     0.164    0.150  0.152
loudness                 0.015   0.682     1.000  ...     0.082    0.050  0.097
speechiness              0.183  -0.029     0.013  ...     0.059    0.065  0.033
acousticness            -0.029  -0.546    -0.372  ...    -0.075   -0.019 -0.114
instrumentalness        -0.002   0.024    -0.154  ...    -0.009   -0.174  0.021
liveness                -0.127   0.164     0.082  ...     1.000   -0.020  0.022
valence                  0.334   0.150     0.050  ...    -0.020    1.000 -0.025
tempo                   -0.185   0.152     0.097  ...     0.022   -0.025  1.000

[9 rows x 9 columns]

Highest positive correlation: energy & loudness = 0.6821
Most negative correlation: energy & acousticness = -0.5459


**Highest positive: `energy` & `loudness` (r = 0.68)**

* Energetic songs (fast, loud, intense) are mixed and mastered louder.
* Spotify's `energy` metric is itself derived partly from perceptual loudness and dynamic range.
* The two features measure overlapping ideas rather than fully independent ones.

**Most negative: `energy` & `acousticness` (r = -0.55)**

* Acoustic tracks (stripped down, unplugged instrumentation) tend to be calmer by nature.
* Heavily produced, distorted, or electronic tracks (high energy) are close to the opposite of a raw acoustic recording.
* Of the 36 feature pairs, this is one of the more intuitive negative relationships.


## High-energy tracks vs overall popularity (NumPy boolean masking)

In [ ]:
energy_arr = df_unique['energy'].to_numpy()
pop_arr = df_unique['track_popularity'].to_numpy()

def high_energy_popularity_comparison(energy_arr, pop_arr):
    mu = energy_arr.mean()
    sigma = energy_arr.std(ddof=1)
    threshold = mu + sigma
    mask = energy_arr > threshold  # NumPy boolean mask, not Pandas filtering
    subset_mean_pop = pop_arr[mask].mean()
    overall_mean_pop = pop_arr.mean()
    return {
        'mu': mu, 'sigma': sigma, 'threshold': threshold,
        'n_subset': int(mask.sum()),
        'subset_mean_popularity': subset_mean_pop,
        'overall_mean_popularity': overall_mean_pop,
        'difference': subset_mean_pop - overall_mean_pop
    }

result_q11 = high_energy_popularity_comparison(energy_arr, pop_arr)
for k, v in result_q11.items():
    print(f"{k}: {v}")

mu: 0.6983875199252364
sigma: 0.18350304473391332
threshold: 0.8818905646591497
n_subset: 4853
subset_mean_popularity: 34.02884813517412
overall_mean_popularity: 39.329771476936095
difference: -5.3009233417619726


* µ (mean energy) is approximately 0.698, and σ is approximately 0.184, so the threshold is approximately 0.882.
* **4,853 tracks** (about 17% of unique tracks) sit above that threshold.
* Their mean `track_popularity` is **34.0**, versus **39.3** overall, a difference of roughly **5.3 points lower**.
* Extremely high energy tracks are, on average, less popular than a typical track, not more.
* This pushes back on the intuitive assumption that "more energy means more popular." Very high energy songs are more likely to sit in niche, high intensity subgenres (hardstyle, aggressive EDM, etc.) than in broadly popular playlists.


# Stage 4: Documentation and AI Tool Reflection

## AI-generated docstrings

**Prompts used** (via Claude, one of this task's accepted AI coding tools):

For `min_max_normalize`:
> Write a NumPy style docstring for this function.
> The function is called min_max_normalize.
> It takes one parameter, arr, a 2D NumPy array with shape (n_samples, n_features).
> It returns a NumPy array of the same shape, with each column independently scaled to the range 0 to 1 using min max scaling.
> Include a one line summary.
> Include a Parameters section describing the array shape and dtype.
> Include a Returns section describing the output shape and value range.
> Do not change the function logic.

For `high_energy_popularity_comparison`:
> Write a NumPy style docstring for this function.
> The function is called high_energy_popularity_comparison.
> It takes two 1D NumPy arrays as parameters, energy_arr and pop_arr, aligned row by row.
> A track counts as high energy when its energy value is greater than the mean energy plus one standard deviation.
> The function returns a dictionary with these exact keys: mu, sigma, threshold, n_subset, subset_mean_popularity, overall_mean_popularity, difference.
> Include a one line summary, a Parameters section, and a Returns section that lists every dictionary key with its meaning and type.
> Do not change the function logic.

Applied to both functions from Stage 3.


In [ ]:
def min_max_normalize(arr: np.ndarray) -> np.ndarray:
    """Scale each column of a 2D array to the [0, 1] range (min-max scaling).

    Parameters
    ----------
    arr : np.ndarray
        2D array of shape (n_samples, n_features).

    Returns
    -------
    np.ndarray
        Array of the same shape, each column rescaled to [0, 1].
    """
    col_min = arr.min(axis=0)
    col_max = arr.max(axis=0)
    return (arr - col_min) / (col_max - col_min)


def high_energy_popularity_comparison(energy_arr: np.ndarray, pop_arr: np.ndarray) -> dict:
    """Compare mean popularity of high-energy tracks against the overall mean.

    "High-energy" is defined as energy > mean(energy) + std(energy).

    Parameters
    ----------
    energy_arr : np.ndarray
        1D array of energy values.
    pop_arr : np.ndarray
        1D array of track_popularity values, aligned with energy_arr.

    Returns
    -------
    dict
        mu, sigma, threshold, n_subset, subset_mean_popularity,
        overall_mean_popularity, and their difference.
    """
    mu = energy_arr.mean()
    sigma = energy_arr.std(ddof=1)
    threshold = mu + sigma
    mask = energy_arr > threshold
    subset_mean_pop = pop_arr[mask].mean()
    overall_mean_pop = pop_arr.mean()
    return {
        'mu': mu, 'sigma': sigma, 'threshold': threshold,
        'n_subset': int(mask.sum()),
        'subset_mean_popularity': subset_mean_pop,
        'overall_mean_popularity': overall_mean_pop,
        'difference': subset_mean_pop - overall_mean_pop
    }

print("Docstrings attached, logic unchanged.")
print(min_max_normalize.__doc__.splitlines()[0])
print(high_energy_popularity_comparison.__doc__.splitlines()[0])

Docstrings attached, logic unchanged.
Scale each column of a 2D array to the [0, 1] range (min-max scaling).
Compare mean popularity of high-energy tracks against the overall mean.


**Evaluation:** used as-is for both.

* The generated docstrings correctly described the actual parameter shapes and dtypes.
* The generated docstrings correctly described the return structure without inventing behaviour the functions do not have.
* Example: the docstring for `min_max_normalize` does not claim it handles NaN values, which is accurate since the function does not handle them.
* No edits were needed beyond checking the output against the real function signatures above.


## Three key insights (for a non-technical reader)

* **Loud and "energetic" are basically the same idea to Spotify's algorithm.** The `energy` and `loudness` scores move together almost every time (r = 0.68). A track rated as high energy is, in practice, almost always a track that was mixed loud.
* **Going too hard can backfire.** The most extreme high-energy tracks (roughly the top 1 in 6) are on average less popular than a typical track, not more. This is likely because ultra high energy styles (hardstyle, aggressive EDM, etc.) appeal to a narrower audience than a mainstream anthem does.
* **Releasing more songs does not make an artist more popular.** Among the artists with the most tracks in the dataset, how many songs they put out barely relates to how well those songs perform (correlation is about 0.13). Quality, marketing, and playlist placement matter far more than sheer volume.


# Bonus 1: Artist Audio Fingerprint

In [ ]:
def artist_fingerprint(data, artist_name):
    artist_tracks = data[data['track_artist'] == artist_name]
    return artist_tracks[feature_cols].to_numpy().mean(axis=0)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# dominant playlist_genre per artist (majority vote, using full df)
artist_genre = df.groupby('track_artist')['playlist_genre'].agg(lambda x: x.value_counts().idxmax())

pairs = [('Kygo', 'Calvin Harris'), ('Drake', 'Don Omar'), ('Queen', 'Martin Garrix')]

print("Cosine similarity on RAW feature scale:")
for a1, a2 in pairs:
    v1 = artist_fingerprint(df_unique, a1)
    v2 = artist_fingerprint(df_unique, a2)
    sim = cosine_similarity(v1, v2)
    print(f"  {a1} ({artist_genre[a1]}) vs {a2} ({artist_genre[a2]}): {sim:.4f}")

Cosine similarity on RAW feature scale:
  Kygo (pop) vs Calvin Harris (pop): 0.9997
  Drake (rap) vs Don Omar (latin): 0.9999
  Queen (rock) vs Martin Garrix (edm): 0.9996


**This result is misleading.**

* All three pairs come out above 0.999 regardless of genre.
* Even Queen (rock) versus Martin Garrix (edm), two artists that do not sound remotely alike, look "almost identical."
* The problem is scale. `loudness` (roughly -4 to -9) and `tempo` (roughly 110 to 130) are numerically enormous compared to features like `danceability` or `acousticness` (0 to 1).
* Cosine similarity is a dot product over vector magnitude, so those two large-magnitude, same-sign dimensions dominate the calculation and drown out every other feature.
* As a result, almost any two artist vectors will look highly similar at this raw scale.

The fix: reuse Stage 3's `min_max_normalize` so every feature contributes on equal footing before comparing.


In [ ]:
df_norm = df_unique.copy()
df_norm[feature_cols] = normalized_array

def artist_fingerprint_norm(data, artist_name):
    return data[data['track_artist'] == artist_name][feature_cols].to_numpy().mean(axis=0)

print("Cosine similarity on NORMALISED [0, 1] feature scale:")
for a1, a2 in pairs:
    v1 = artist_fingerprint_norm(df_norm, a1)
    v2 = artist_fingerprint_norm(df_norm, a2)
    sim = cosine_similarity(v1, v2)
    print(f"  {a1} ({artist_genre[a1]}) vs {a2} ({artist_genre[a2]}): {sim:.4f}")

Cosine similarity on NORMALISED [0, 1] feature scale:
  Kygo (pop) vs Calvin Harris (pop): 0.9920
  Drake (rap) vs Don Omar (latin): 0.9721
  Queen (rock) vs Martin Garrix (edm): 0.9527


* With normalised features the similarities finally spread out (0.992, 0.972, 0.953) instead of clustering at about 1.0.
* The ranking loosely tracks genre: **Kygo vs Calvin Harris** (both pop-leaning) score highest, and **Queen vs Martin Garrix** (rock vs edm, the most different pairing tested) score lowest.
* There is partial support for "same genre gives higher similarity," but it is a weak signal, not a clean rule. **Drake vs Don Omar** (rap vs latin, different genres) still scores fairly high at 0.972.
* This is expected. `artist_fingerprint` only summarises 9 numeric audio production features (tempo, energy, danceability, etc.).
* It has no information about language, vocal style, lyrical content, or instrumentation choice, all of which drive a lot of what actually makes rap and latin feel like different genres to a listener.
* Two artists can have a genuinely similar production sound while still belonging to different genre labels for reasons this feature set cannot see.


# Bonus 2: Genre Cluster Profile

## Genre centroids

In [ ]:
# Genre centroids use the full df (not df_unique). A track's placement in a
# genre's playlist is a legitimate signal for that genre's audio profile,
# consistent with how Stage 2's per-genre groupby was justified.
genres = sorted(df['playlist_genre'].unique())
centroid_matrix = df.groupby('playlist_genre')[feature_cols].mean().loc[genres].to_numpy()

print(pd.DataFrame(centroid_matrix, index=genres, columns=feature_cols).round(3))
print()
print("centroid_matrix shape:", centroid_matrix.shape)

       danceability  energy  loudness  ...  liveness  valence    tempo
edm           0.655   0.802    -5.427  ...     0.212    0.401  125.768
latin         0.713   0.708    -6.264  ...     0.181    0.606  118.622
pop           0.639   0.701    -6.315  ...     0.177    0.504  120.743
r&b           0.670   0.591    -7.865  ...     0.175    0.531  114.222
rap           0.718   0.651    -7.042  ...     0.192    0.505  120.655
rock          0.521   0.733    -7.589  ...     0.203    0.537  124.989

[6 rows x 9 columns]

centroid_matrix shape: (6, 9)


## Pairwise genre distance matrix (vectorised, no loops)

In [ ]:
# Broadcasting: (n_genres, 1, 9) - (1, n_genres, 9) -> (n_genres, n_genres, 9)
diff = centroid_matrix[:, np.newaxis, :] - centroid_matrix[np.newaxis, :, :]
dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))

print(pd.DataFrame(dist_matrix, index=genres, columns=genres).round(3))

          edm  latin    pop     r&b    rap    rock
edm     0.000  7.202  5.108  11.806  5.370   2.313
latin   7.202  0.000  2.126   4.685  2.182   6.507
pop     5.108  2.126  0.000   6.704  0.749   4.435
r&b    11.806  4.685  6.704   0.000  6.487  10.773
rap     5.370  2.182  0.749   6.487  0.000   4.376
rock    2.313  6.507  4.435  10.773  4.376   0.000


In [ ]:
n = len(genres)
mask_offdiag = ~np.eye(n, dtype=bool)
tri_mask = np.triu(mask_offdiag)
dist_masked = np.where(tri_mask, dist_matrix, np.nan)

min_idx = np.unravel_index(np.nanargmin(dist_masked), dist_masked.shape)
max_idx = np.unravel_index(np.nanargmax(dist_masked), dist_masked.shape)

print(f"Most similar genres: {genres[min_idx[0]]} & {genres[min_idx[1]]} (distance = {dist_matrix[min_idx]:.4f})")
print(f"Most different genres: {genres[max_idx[0]]} & {genres[max_idx[1]]} (distance = {dist_matrix[max_idx]:.4f})")

Most similar genres: pop & rap (distance = 0.7491)
Most different genres: edm & r&b (distance = 11.8059)


Mostly matches musical intuition.

* **pop & rap** land closest together. Both are mainstream, moderately danceable, moderate tempo, production heavy genres in a 2020 Spotify playlist context, so their average audio fingerprint converges even though the genres sound different to a listener. This profile only sees the 9 numeric audio features, not vocals or lyrics.
* **edm & r&b** are the most different. edm centroids are high on energy, loudness, and tempo with low acousticness, while r&b is the opposite: lower energy, quieter, more acoustic leaning, and slower.
* That contrast lines up well with how those genres are actually described.


## Genre recommender

In [ ]:
def recommend_similar_genre(genre, top_k=3):
    idx = genres.index(genre)
    distances = dist_matrix[idx]
    order = np.argsort(distances)
    order = order[order != idx]  # exclude the genre itself
    top = order[:top_k]
    return [(genres[i], round(float(distances[i]), 4)) for i in top]

print("Genres closest to 'pop':", recommend_similar_genre('pop', 3))
print("Genres closest to 'rock':", recommend_similar_genre('rock', 3))

Genres closest to 'pop': [('rap', 0.7491), ('latin', 2.1258), ('rock', 4.4345)]
Genres closest to 'rock': [('edm', 2.313), ('rap', 4.3761), ('pop', 4.4345)]


# Bonus 3: Reusable Analysis Class with AI Documentation

**Prompt used** (via Claude):

> Write NumPy style docstrings for every method in this Python class called SpotifyAnalyzer.
> `__init__` takes a CSV url string and loads it into a DataFrame.
> `clean` drops full row duplicates and builds a track_id unique DataFrame, returning the cleaned DataFrame.
> `summary_statistics` returns a DataFrame with mean, median, standard deviation, and IQR for every numeric column.
> `genre_analysis` groups by playlist_genre and returns mean, standard deviation, and median for track_popularity, danceability, energy, and valence.
> `numpy_feature_analysis` returns a dictionary with keys normalized_array, correlation_matrix, highest_positive_pair, most_negative_pair.
> `genre_clustering` returns a dictionary with keys genres, centroid_matrix, distance_matrix, most_similar_pair, most_different_pair.
> `generate_report` returns a dictionary summarising the key findings from the other methods.
> For each method, include a one line summary, a Parameters section if the method takes arguments beyond self, and a Returns section describing the exact type and structure of the return value.
> Do not change any method logic.

Applied to the whole `SpotifyAnalyzer` class below.


In [ ]:
from typing import Optional
import pandas as pd
import numpy as np


class SpotifyAnalyzer:
    """
    End-to-end EDA pipeline for the TidyTuesday Spotify Songs dataset.

    Wraps the loading, cleaning, genre analysis, and NumPy-based audio
    feature analysis performed earlier in this notebook into a single
    reusable class.
    """

    FEATURE_COLS = ['danceability', 'energy', 'loudness', 'speechiness',
                     'acousticness', 'instrumentalness', 'liveness',
                     'valence', 'tempo']

    def __init__(self, url: str) -> None:
        """
        Load the dataset from `url` and store it as a DataFrame.

        Parameters
        ----------
        url : str
            CSV URL to load with pandas.read_csv.
        """
        self.df: pd.DataFrame = pd.read_csv(url)
        self.df_unique: Optional[pd.DataFrame] = None
        self._report: dict = {}

    def clean(self) -> pd.DataFrame:
        """
        Drop full-row duplicates and build a track_id-unique view.

        Returns
        -------
        pd.DataFrame
            The de-duplicated (full-row) DataFrame.
        """
        self.df = self.df.drop_duplicates()
        self.df_unique = self.df.drop_duplicates(
            subset=['track_id'], keep='first'
        ).reset_index(drop=True)
        return self.df

    def summary_statistics(self) -> pd.DataFrame:
        """
        Compute mean, median, standard deviation, and IQR for every
        numeric column, with IQR calculated manually via NumPy percentiles.

        Returns
        -------
        pd.DataFrame
            One row per numeric column with the four summary metrics.
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        rows = []
        for col in numeric_cols:
            data = self.df[col].dropna().to_numpy()
            q75, q25 = np.percentile(data, [75, 25])
            rows.append({
                'column': col,
                'mean': np.mean(data),
                'median': np.median(data),
                'std': np.std(data, ddof=1),
                'iqr': q75 - q25
            })
        return pd.DataFrame(rows)

    def genre_analysis(self) -> pd.DataFrame:
        """
        Compute mean/std/median of track_popularity, danceability, energy,
        and valence grouped by playlist_genre.

        Returns
        -------
        pd.DataFrame
            Multi-level column DataFrame indexed by playlist_genre.
        """
        cols = ['track_popularity', 'danceability', 'energy', 'valence']
        return self.df.groupby('playlist_genre')[cols].agg(['mean', 'std', 'median'])

    def numpy_feature_analysis(self) -> dict:
        """
        Normalise the nine audio features (min-max, vectorised) and compute
        their correlation matrix using df_unique.

        Returns
        -------
        dict
            Keys: 'normalized_array', 'correlation_matrix',
            'highest_positive_pair', 'most_negative_pair'.
        """
        if self.df_unique is None:
            self.clean()

        arr = self.df_unique[self.FEATURE_COLS].to_numpy()
        normalized = (arr - arr.min(axis=0)) / (arr.max(axis=0) - arr.min(axis=0))

        corr = np.corrcoef(arr, rowvar=False)
        mask = np.triu(~np.eye(len(self.FEATURE_COLS), dtype=bool))
        corr_masked = np.where(mask, corr, np.nan)
        max_idx = np.unravel_index(np.nanargmax(corr_masked), corr_masked.shape)
        min_idx = np.unravel_index(np.nanargmin(corr_masked), corr_masked.shape)

        return {
            'normalized_array': normalized,
            'correlation_matrix': corr,
            'highest_positive_pair': (
                self.FEATURE_COLS[max_idx[0]], self.FEATURE_COLS[max_idx[1]],
                float(corr[max_idx])
            ),
            'most_negative_pair': (
                self.FEATURE_COLS[min_idx[0]], self.FEATURE_COLS[min_idx[1]],
                float(corr[min_idx])
            )
        }

    def genre_clustering(self) -> dict:
        """
        Build per-genre centroids over the nine audio features and compute
        the pairwise Euclidean distance matrix between genres.

        Returns
        -------
        dict
            Keys: 'genres', 'centroid_matrix', 'distance_matrix',
            'most_similar_pair', 'most_different_pair'.
        """
        genres = sorted(self.df['playlist_genre'].unique())
        centroid_matrix = (
            self.df.groupby('playlist_genre')[self.FEATURE_COLS]
            .mean().loc[genres].to_numpy()
        )
        diff = centroid_matrix[:, np.newaxis, :] - centroid_matrix[np.newaxis, :, :]
        dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))

        n = len(genres)
        mask = np.triu(~np.eye(n, dtype=bool))
        dist_masked = np.where(mask, dist_matrix, np.nan)
        min_idx = np.unravel_index(np.nanargmin(dist_masked), dist_masked.shape)
        max_idx = np.unravel_index(np.nanargmax(dist_masked), dist_masked.shape)

        return {
            'genres': genres,
            'centroid_matrix': centroid_matrix,
            'distance_matrix': dist_matrix,
            'most_similar_pair': (genres[min_idx[0]], genres[min_idx[1]], float(dist_matrix[min_idx])),
            'most_different_pair': (genres[max_idx[0]], genres[max_idx[1]], float(dist_matrix[max_idx]))
        }

    def generate_report(self) -> dict:
        """
        Run the full pipeline and collect all key findings into a single
        JSON-serialisable dictionary.

        Returns
        -------
        dict
            Summary of dataset shape, top-variance genre, correlation
            highlights, and genre similarity highlights.
        """
        self.clean()
        genre_stats = self.genre_analysis()
        feature_results = self.numpy_feature_analysis()
        cluster_results = self.genre_clustering()

        top_variance_genre = genre_stats[('track_popularity', 'std')].idxmax()

        self._report = {
            'n_rows': int(len(self.df)),
            'n_unique_tracks': int(len(self.df_unique)),
            'genre_with_highest_popularity_variance': top_variance_genre,
            'highest_positive_correlation': feature_results['highest_positive_pair'],
            'most_negative_correlation': feature_results['most_negative_pair'],
            'most_similar_genres': cluster_results['most_similar_pair'],
            'most_different_genres': cluster_results['most_different_pair'],
        }
        return self._report


# ---- demonstration: runs entirely from this single cell ----
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-01-21/spotify_songs.csv'
analyzer = SpotifyAnalyzer(url)
report = analyzer.generate_report()

import json as _json
print(_json.dumps(report, indent=2))

{
  "n_rows": 32833,
  "n_unique_tracks": 28356,
  "genre_with_highest_popularity_variance": "r&b",
  "highest_positive_correlation": [
    "energy",
    "loudness",
    0.6821377424910235
  ],
  "most_negative_correlation": [
    "energy",
    "acousticness",
    -0.5458861907892343
  ],
  "most_similar_genres": [
    "pop",
    "rap",
    0.7491492326854329
  ],
  "most_different_genres": [
    "edm",
    "r&b",
    11.805934411817171
  ]
}


`generate_report()` reproduces the headline numbers from earlier in the notebook.

* R&B has the highest popularity variance.
* energy and loudness form the strongest positive correlation.
* energy and acousticness form the strongest negative correlation.
* pop and rap are the most similar genres.
* edm and r&b are the most different genres.

This confirms the class refactor did not silently change any of the underlying logic. It just packaged the existing analysis for reuse.
